# 11 — Evaluations

Runs an LLM-judged evaluation suite against a **ReAct agent**.

Architecture:
- `EvalDataset` — a list of `EvalCase` (input + expected output + tags)
- `EvalRunner` — runs the agent for every case with parallelism + timeout
- `LLMJudge` — scores each response against multiple criteria in parallel
- `CostTracker` — hooks into the LLM lifecycle to measure token costs
- Reports exported as **JSON** and **Markdown**

**Prerequisites**: `OPENAI_API_KEY` set and `results/` directory writable.

In [1]:
import os, asyncio
os.makedirs('results', exist_ok=True)
print('Results dir ready')

Results dir ready


In [2]:
from agent_framework.core.agents.react_agent import ReActAgent
from agent_framework.providers.llm.openai.openai_client import OpenAIClient
from agent_framework.core.tools.builtin_tools import CalculatorTool, GetCurrentTimeTool
from agent_framework.core.memory.unbounded_memory import UnboundedMemory
from agent_framework.core.context.implementations import UnboundedContext
from agent_framework.core.hooks import HookEvent, HookManager, CostTracker
from agent_framework.evals import (
    EvalCase, EvalDataset, EvalRunner, LLMJudge,
    CORRECTNESS, HELPFULNESS, SAFETY, CONCISENESS,
)

## Define an evaluation dataset

In [3]:
dataset = EvalDataset(
    name='basic_math_qa',
    description='Basic math and tool-use tests',
    cases=[
        EvalCase(input='What is 15 * 7?', expected_output='105', tags=['math']),
        EvalCase(input='Calculate the square root of 144', expected_output='12', tags=['math']),
        EvalCase(input='What is 2^10?', expected_output='1024', tags=['math']),
        EvalCase(input='What is the current time?', expected_output=None, tags=['tools']),
        EvalCase(input='If a train travels 60 mph for 2.5 hours, how far does it go?', expected_output='150 miles', tags=['word_problem']),
    ],
)
print(f'Dataset: {dataset.name} ({dataset.size} cases)')

Dataset: basic_math_qa (5 cases)


## Build agent, judge, and run evaluation

In [4]:
async def run_eval():
    hooks = HookManager()
    cost_tracker = CostTracker(model='gpt-4.1-nano')
    hooks.register(HookEvent.LLM_END, cost_tracker.on_llm_end)
    hooks.register(HookEvent.RUN_END, cost_tracker.on_run_end)

    agent = ReActAgent(
        name='eval-agent',
        description='Agent under evaluation',
        model_client=OpenAIClient(model='gpt-4.1-nano'),
        tools=[CalculatorTool(), GetCurrentTimeTool()],
        memory=UnboundedMemory(),
        model_context=UnboundedContext(),
        hooks=hooks,
        max_iterations=5,
        verbose=False,
    )

    judge = LLMJudge(
        model_client=OpenAIClient(model='gpt-4.1-mini'),
        criteria=[CORRECTNESS, HELPFULNESS, SAFETY, CONCISENESS],
        parallel=True,
    )

    runner = EvalRunner(agent=agent, judge=judge, concurrency=1, case_timeout=60.0, reset_agent=True)

    print('Running evaluation...')
    report = await runner.run(dataset)
    print(report.summary())

    EvalRunner.export_json(report, 'results/eval_report.json')
    EvalRunner.export_markdown(report, 'results/eval_report.md')
    print('Reports saved to results/')
    print(f'Cost: {cost_tracker.stats}')

await run_eval()

Running evaluation...
Starting eval: basic_math_qa (5 cases, concurrency=1)
Run cost: $0.000048 (351 prompt + 33 completion tokens, 2 calls)
  [1/5] PASS (avg=0.88) What is 15 * 7?...
Run cost: $0.000148 (993 prompt + 121 completion tokens, 5 calls)
  [2/5] PASS (avg=0.94) Calculate the square root of 144...
Run cost: $0.000194 (1341 prompt + 150 completion tokens, 7 calls)
  [3/5] PASS (avg=0.88) What is 2^10?...
Run cost: $0.000255 (1710 prompt + 209 completion tokens, 9 calls)
  [4/5] FAIL (avg=0.75) What is the current time?...
Run cost: $0.000328 (2152 prompt + 282 completion tokens, 11 calls)
  [5/5] PASS (avg=1.00) If a train travels 60 mph for 2.5 hours, how far d...
Eval complete:
  EVAL REPORT: basic_math_qa
  Agent: eval-agent | Model: gpt-4.1-nano
  Cases:     5
  Passed:    4 (80.0%)
  Failed:    1
  Errors:    0
  Avg Score: 0.887
  Avg Latency: 2.68s
  Total Tokens: 2,434
------------------------------------------------------------
  Scores by Criterion:
    correctness 